In [6]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!pip install -q pymupdf

In [8]:
import json
from pathlib import Path
import pandas as pd
import fitz  # PyMuPDF

PROJECT_DIR = Path(
    "/content/drive/MyDrive/NLP Project"
)

PAPER_DIR = PROJECT_DIR / "papers"

METADATA_FILE = PROJECT_DIR / "corpus_inventory.csv"



# Separate output folders
DATA_DIR = PROJECT_DIR / "data"
EXTRACTED_DIR = DATA_DIR / "extracted_text"

OUTPUT_FILE = EXTRACTED_DIR / "extracted_pages.jsonl"
ERRORS_FILE = DATA_DIR / "extraction_errors.json"
OUTPUT_INVENTORY = PROJECT_DIR / "corpus_inventory_extracted.csv"

metadata = pd.read_csv(METADATA_FILE)
metadata.head()

,article_id,title,author,year,venue,doi_or_url,topical_category,license_or_access,pdf_path
0,FD_001,Credit card fraud detection in the era of disr...,"Asma Cherif, Arwa Badhib, Heyfa Ammar, Suhair ...",2023,Journal of King Saud University-Computer and I...,https://doi.org/10.1016/j.jksuci.2022.11.008,"Credit card fraud detection, machine learning,...",open access - CC BY-NC-ND 4.0,/content/drive/MyDrive/NLP Project/papers/Sona...
1,FD_002,Credit Card Fraud Detection using Machine Lear...,"V. N. Dornadula, S. Geetha",2020,"Procedia Computer Science, Volume 165, pages 6...",https://doi.org/10.1016/j.procs.2020.01.057,"Credit card fraud detection, machine learning,...",Open access - CC BY-NC-ND 4.0,/content/drive/MyDrive/NLP Project/papers/Sona...
2,FD_003,Credit Card Fraud Detection Using AdaBoost and...,"Kuldeep Randhawa, Chu Kiong Loo, Manjeevan See...",2018,"IEEE Access, Volume 6, pages 14277–14284",https://doi.org/10.1109/ACCESS.2018.2806420,"Credit card fraud detection, machine learning,...",Open access - CC BY 4.0,/content/drive/MyDrive/NLP Project/papers/Sona...
3,FD_004,Credit Card Fraud Detection via Intelligent Sa...,"Chiao-Ting Chen, Chi Lee, Szu-Hao Huang, Wen-C...",2024,ACM Transactions on Intelligent Systems and Te...,https://doi.org/10.1145/3641283,"Credit card fraud detection, transaction fraud...",Free access,/content/drive/MyDrive/NLP Project/papers/Sona...
4,FD_005,Credit Card Fraud Detection with Automated Mac...,"Vasilios Plakandaras, Periklis Gogas, Theophil...",2022,"Applied Artificial Intelligence, Volume 36, Is...",https://doi.org/10.1080/08839514.2022.2086354,"Credit card fraud detection, automated machine...",Free access,/content/drive/MyDrive/NLP Project/papers/Sona...


In [9]:
metadata['doi_or_url'].duplicated().sum()


np.int64(19)

In [10]:

metadata = metadata.drop_duplicates(subset=['doi_or_url'], keep='first').reset_index(drop=True)


In [11]:
metadata

,article_id,title,author,year,venue,doi_or_url,topical_category,license_or_access,pdf_path
0,FD_001,Credit card fraud detection in the era of disr...,"Asma Cherif, Arwa Badhib, Heyfa Ammar, Suhair ...",2023,Journal of King Saud University-Computer and I...,https://doi.org/10.1016/j.jksuci.2022.11.008,"Credit card fraud detection, machine learning,...",open access - CC BY-NC-ND 4.0,/content/drive/MyDrive/NLP Project/papers/Sona...
1,FD_002,Credit Card Fraud Detection using Machine Lear...,"V. N. Dornadula, S. Geetha",2020,"Procedia Computer Science, Volume 165, pages 6...",https://doi.org/10.1016/j.procs.2020.01.057,"Credit card fraud detection, machine learning,...",Open access - CC BY-NC-ND 4.0,/content/drive/MyDrive/NLP Project/papers/Sona...
2,FD_003,Credit Card Fraud Detection Using AdaBoost and...,"Kuldeep Randhawa, Chu Kiong Loo, Manjeevan See...",2018,"IEEE Access, Volume 6, pages 14277–14284",https://doi.org/10.1109/ACCESS.2018.2806420,"Credit card fraud detection, machine learning,...",Open access - CC BY 4.0,/content/drive/MyDrive/NLP Project/papers/Sona...
3,FD_004,Credit Card Fraud Detection via Intelligent Sa...,"Chiao-Ting Chen, Chi Lee, Szu-Hao Huang, Wen-C...",2024,ACM Transactions on Intelligent Systems and Te...,https://doi.org/10.1145/3641283,"Credit card fraud detection, transaction fraud...",Free access,/content/drive/MyDrive/NLP Project/papers/Sona...
4,FD_005,Credit Card Fraud Detection with Automated Mac...,"Vasilios Plakandaras, Periklis Gogas, Theophil...",2022,"Applied Artificial Intelligence, Volume 36, Is...",https://doi.org/10.1080/08839514.2022.2086354,"Credit card fraud detection, automated machine...",Free access,/content/drive/MyDrive/NLP Project/papers/Sona...
...,...,...,...,...,...,...,...,...,...
177,FD_196,A Machine Learning and Explainable Artificial ...,"Diana Olivia, Zaid Khan, Sucharitha Shetty",2025,Inteligencia Artificial,https://doi.org/10.4114/intartif.vol28iss75pp1...,Insurance fraud; explainable AI,Open Access,/content/drive/MyDrive/NLP Project/papers/Chek...
178,FD_197,Application Technologies and Challenges of Big...,Haoran Jiang,2024,Open Journal of Applied Sciences,https://doi.org/10.4236/ojapps.2024.1411213,AML; big data; fraud detection,Open Access - CC BY 4.0,/content/drive/MyDrive/NLP Project/papers/Chek...
179,FD_199,Graph convolution network for fraud detection ...,"Ahmad Asiri, K. Somasundaram",2025,Scientific Reports,https://doi.org/10.1038/s41598-025-95672-w,"Cryptocurrency or blockchain fraud detection, ...",Open Access - CC BY 4.0,/content/drive/MyDrive/NLP Project/papers/tshe...
180,FD_200,Detecting Fraudulent Transactions for Differen...,"Shaziya Islam, Gagan Raj Gupta, Apu Chakrabort...",2025,Human-Centric Intelligent Systems,https://doi.org/10.1007/s44230-025-00097-3,"Financial fraud detection, graph neural networks",Open Access - CC BY 4.0,/content/drive/MyDrive/NLP Project/papers/tshe...


In [12]:
def extract_pdf(pdf_path):
    pages = []
    with fitz.open(str(pdf_path)) as doc:
        for i, page in enumerate(doc, start=1):
            text = page.get_text("text").strip()
            pages.append({
                "page_number": i,
                "page_label": page.get_label() or str(i),
                "text": text,
                "n_chars": len(text),
            })
    return pages

In [13]:
errors, total_pages = [], 0

with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    for idx, row in metadata.iterrows():
        article_id = row["article_id"]
        pdf_path = Path(row["pdf_path"])
        try:
            if not pdf_path.is_file():
                raise FileNotFoundError(f"PDF not found: {pdf_path}")

            pages = extract_pdf(pdf_path)
            if not pages:
                raise ValueError("No pages extracted")

            for p in pages:
                out.write(json.dumps({
                    "article_id": article_id,
                    "title": row["title"],
                    "year": int(row["year"]),
                    **p
                }, ensure_ascii=False) + "\n")

            metadata.at[idx, "ingestion_status"] = "Success"
            metadata.at[idx, "extraction_status"] = "Success"
            metadata.at[idx, "n_pages"] = len(pages)
            metadata.at[idx, "avg_chars_per_page"] = round(sum(p["n_chars"] for p in pages) / len(pages))
            total_pages += len(pages)

        except Exception as e:
            metadata.at[idx, "ingestion_status"] = "Success" if pdf_path.is_file() else "Failed"
            metadata.at[idx, "extraction_status"] = "Failed"
            errors.append({"article_id": article_id, "error": str(e)})
            print(f"Failed: {article_id} - {e}")

metadata.to_csv(OUTPUT_INVENTORY, index=False)
with open(ERRORS_FILE, "w", encoding="utf-8") as f:
    json.dump(errors, f, indent=2, ensure_ascii=False)

print("Total papers:", len(metadata))
print("Extracted OK:", (metadata["extraction_status"] == "Success").sum())
print("Total pages: ", total_pages)
print("Failed:", len(errors))

Total papers: 182
Extracted OK: 182
Total pages:  3619
Failed: 0


In [16]:
pages_df = pd.read_json(OUTPUT_FILE, lines=True)
print("Pages:", len(pages_df), "| Articles:", pages_df["article_id"].nunique())

def show_page(article_id, page_number, max_chars=2000):
    row = pages_df[(pages_df["article_id"] == article_id) &
                   (pages_df["page_number"] == page_number)].iloc[0]
    print(f"{row['article_id']} | {row['title']}")
    print(f"PDF page {row['page_number']} (printed label: {row['page_label']}) | {row['n_chars']} chars\n")
    print(row["text"][:max_chars])

show_page("FD_001", 1)

Pages: 3619 | Articles: 182
FD_001 | Credit card fraud detection in the era of disruptive technologies, A systematic review.
PDF page 1 (printed label: 145) | 6031 chars

Credit card fraud detection in the era of disruptive technologies:
A systematic review
Asma Cherif a,b, Arwa Badhib a, Heyfa Ammar b,c, Suhair Alshehri a, Manal Kalkatawi a, Abdessamad Imine d
a Department of Information Technology, Faculty of Computing & Information Technology, King Abdulaziz University, Jeddah, Saudi Arabia
b Center of Excellence in Smart Environment Research, King Abdulaziz University, Jeddah, Saudi Arabia
c RISC-ENIT Research Lab, National Engineering School of Tunis, Universty of Almanar, Tunisia
d Lorraine University, CNRS, INRIA, Vandœuvre-lès-Nancy, France
a r t i c l e
i n f o
Article history:
Received 11 July 2022
Revised 3 November 2022
Accepted 16 November 2022
Available online 5 December 2022
Keywords:
Credit card fraud detection
Machine learning
Deep learning
Class imbalance
a b s t r a 